# Segmentation des Ventes Supermarché
## Analyse par K-Means Clustering
**Projet Data Science — L3/M1**

---
## 1. Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

print('Bibliothèques importées avec succès !')

---
## 2. Chargement des données

In [ ]:
df = pd.read_csv('supermarket_sales.csv')
print('Shape du dataset :', df.shape)
df.head()

In [ ]:
# Afficher les colonnes disponibles
print('Colonnes :', df.columns.tolist())

In [ ]:
# Informations générales
df.info()

In [ ]:
# Statistiques descriptives
df.describe()

---
## 3. Prétraitement des données

In [ ]:
# Vérifier les valeurs manquantes
print('Valeurs manquantes par colonne :')
print(df.isnull().sum())

In [ ]:
# Vérifier les doublons
print('Nombre de doublons :', df.duplicated().sum())

In [ ]:
# Convertir la colonne Date
df['Date'] = pd.to_datetime(df['Date'])
print('Date min :', df['Date'].min())
print('Date max :', df['Date'].max())

In [ ]:
# Sélectionner les variables numériques pour le clustering
features = ['Unit price', 'Quantity', 'Total', 'Rating']

X = df[features].copy()
print('Variables sélectionnées :')
print(X.describe())

In [ ]:
# Normalisation des données avec StandardScaler
# Nécessaire pour que K-Means ne soit pas biaisé par les échelles
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Données normalisées — moyenne :', X_scaled.mean().round(2))
print('Données normalisées — écart-type :', X_scaled.std().round(2))

---
## 4. Analyse exploratoire (EDA)

In [ ]:
# Distribution du chiffre d'affaires
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(df['Total'], bins=30, color='orange', edgecolor='black')
plt.title('Distribution du Total des ventes')
plt.xlabel('Total ($)')
plt.ylabel('Fréquence')

plt.subplot(1, 2, 2)
plt.hist(df['Quantity'], bins=10, color='steelblue', edgecolor='black')
plt.title('Distribution des Quantités')
plt.xlabel('Quantité')
plt.ylabel('Fréquence')

plt.tight_layout()
plt.show()

In [ ]:
# CA par ligne de produit
ca_produit = df.groupby('Product line')['Total'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
ca_produit.plot(kind='bar', color='orange', edgecolor='black')
plt.title('Chiffre d\'affaires par ligne de produit')
plt.xlabel('Ligne de produit')
plt.ylabel('CA Total ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation
plt.figure(figsize=(7, 5))
sns.heatmap(X.corr(), annot=True, cmap='YlOrRd', fmt='.2f')
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.show()

---
## 5. Choix du nombre de clusters — Méthode du coude

In [ ]:
# Tester K de 2 à 8 et calculer l'inertie
inertias = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

# Afficher la courbe du coude
plt.figure(figsize=(8, 5))
plt.plot(list(K_range), inertias, marker='o', color='orange', linewidth=2)
plt.title('Méthode du coude — Choix optimal de K')
plt.xlabel('Nombre de clusters K')
plt.ylabel('Inertie')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Score Silhouette pour valider le choix de K
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_tmp = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels_tmp)
    sil_scores.append(score)
    print(f'K={k} → Score Silhouette : {score:.4f}')

best_k = list(K_range)[sil_scores.index(max(sil_scores))]
print(f'\nMeilleur K selon Silhouette : {best_k}')

---
## 6. Entraînement du modèle K-Means

In [ ]:
# Entraînement avec K=3 (ajuster selon la méthode du coude)
K = 3

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(X_scaled)

print(f'Modèle K-Means entraîné avec K={K}')
print(f'Inertie finale : {kmeans.inertia_:.2f}')

---
## 7. Prédictions — Assignation des clusters

In [ ]:
# Prédire le cluster de chaque transaction
df['Cluster'] = kmeans.predict(X_scaled)

print('Nombre de transactions par cluster :')
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Afficher un exemple de prédiction pour une nouvelle transaction
nouvelle_transaction = pd.DataFrame({
    'Unit price': [55.0],
    'Quantity': [4],
    'Total': [231.0],
    'Rating': [7.5]
})

# Normaliser et prédire
nouvelle_scaled = scaler.transform(nouvelle_transaction)
cluster_predit = kmeans.predict(nouvelle_scaled)

print(f'La nouvelle transaction appartient au Cluster : {cluster_predit[0]}')

---
## 8. Analyse des clusters

In [ ]:
# Profil moyen de chaque cluster
profil_clusters = df.groupby('Cluster')[features].mean().round(2)
print('Profil moyen par cluster :')
profil_clusters

In [ ]:
# Visualisation des clusters (Total vs Quantity)
colors = ['orange', 'steelblue', 'green', 'red', 'purple']

plt.figure(figsize=(9, 6))
for i in range(K):
    cluster_data = df[df['Cluster'] == i]
    plt.scatter(cluster_data['Total'], cluster_data['Quantity'],
                label=f'Cluster {i}', color=colors[i], alpha=0.6)

plt.title('Segmentation K-Means : Total vs Quantité')
plt.xlabel('Total ($)')
plt.ylabel('Quantité')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visualisation PCA (réduction en 2D pour mieux voir les clusters)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
for i in range(K):
    mask = df['Cluster'] == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                label=f'Cluster {i}', color=colors[i], alpha=0.6)

plt.title('Projection PCA des clusters')
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f'Variance expliquée par les 2 composantes : {pca.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# CA total par cluster
ca_cluster = df.groupby('Cluster')['Total'].sum().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
ca_cluster.plot(kind='bar', color=['orange', 'steelblue', 'green'], edgecolor='black')
plt.title('Chiffre d\'affaires total par cluster')
plt.xlabel('Cluster')
plt.ylabel('CA Total ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 9. Interprétation des clusters

In [ ]:
# Résumé complet des clusters
resume = df.groupby('Cluster').agg(
    Nb_transactions=('Total', 'count'),
    CA_total=('Total', 'sum'),
    Panier_moyen=('Total', 'mean'),
    Quantite_moyenne=('Quantity', 'mean'),
    Note_moyenne=('Rating', 'mean')
).round(2)

print('Résumé des clusters :')
resume

In [ ]:
# Interprétation métier
total_moyen = df['Total'].mean()

for c in sorted(df['Cluster'].unique()):
    panier = df[df['Cluster'] == c]['Total'].mean()
    n = (df['Cluster'] == c).sum()
    
    if panier >= total_moyen * 1.2:
        profil = 'Segment Premium — fort panier, clients à fidéliser'
    elif panier <= total_moyen * 0.8:
        profil = 'Segment à Activer — panier faible, besoin de promotions'
    else:
        profil = 'Segment Intermédiaire — comportement moyen, potentiel de cross-selling'
    
    print(f'Cluster {c} ({n} transactions) → {profil}')

---
## 10. Export des résultats

In [ ]:
# Sauvegarder le dataset avec les clusters
df.to_csv('supermarket_sales_segmente.csv', index=False)
print('Fichier exporté : supermarket_sales_segmente.csv')
print(f'Shape final : {df.shape}')
df[['Invoice ID', 'Total', 'Quantity', 'Rating', 'Cluster']].head(10)